# Employment Analysis
Monthly workforce snapshots from Jan 2024 – Feb 2026. Used as the denominator for computing true separation rates and establishing pre-DOGE agency baselines.

## Load Data

In [1]:
from pathlib import Path
import sys

sys.path.append(str(Path.cwd().parent))
from scripts.data_loader import load_r2_data
import pandas as pd
import numpy as np

### NOTE: This will take half an hour!

In [ ]:
# Load all employment snapshots (2024-2026), aggregated per file to keep memory flat
AGG_COLS = [
    "agency_subelement_code",
    "agency_subelement",
    "age_bracket",
    "tenure_code",
    "appointment_type_code",
]

e_df = load_r2_data("employment", agg_cols=AGG_COLS)

print(f"Shape: {e_df.shape[0]:,} rows x {e_df.shape[1]} columns")
print(f"Months covered: {sorted(e_df['year'].astype(str) + '-' + e_df['month'].astype(str).str.zfill(2).unique().tolist())}")

Cache the DF. 

In [ ]:
e_df.to_parquet("employment.parquet")

In [ ]:
e_df['ym'] = e_df['year'].astype(str) + '-' + e_df['month'].astype(str).str.zfill(2)
print("Months available:", sorted(e_df['ym'].unique()))

## Total Workforce Size Over Time
Sum `count` across all rows for each month to get total headcount.

In [ ]:
# Total federal workforce per month
monthly_total = (
    e_df.groupby(['year', 'month', 'ym'])['count']
    .sum()
    .reset_index()
    .sort_values('ym')
)
monthly_total

## Agency-Level Headcount Over Time
Track headcount per agency subelement month by month to identify workforce collapse.

In [ ]:
# Headcount per agency subelement per month
agency_monthly = (
    e_df.groupby(['ym', 'agency_subelement_code', 'agency_subelement'])['count']
    .sum()
    .reset_index()
    .sort_values(['agency_subelement_code', 'ym'])
)

# Pivot to wide: rows=agency, cols=month
agency_pivot = agency_monthly.pivot_table(
    index=['agency_subelement_code', 'agency_subelement'],
    columns='ym',
    values='count',
    fill_value=0
)
agency_pivot.head(10)

## Pre-DOGE Baseline Workforce Characteristics
Use Jan–Jun 2025 as the pre-DOGE baseline (before executive orders took effect in practice).
Capture agency-level profile: size, age mix, tenure mix, % temporary appointments.

In [ ]:
# Pre-DOGE snapshot: average of Jan-Jun 2025
pre_doge = e_df[(e_df['year'] == 2025) & (e_df['month'] <= 6)].copy()

# Total headcount per agency in pre-DOGE period (average monthly)
pre_doge_size = (
    pre_doge.groupby(['agency_subelement_code', 'agency_subelement', 'ym'])['count']
    .sum()
    .groupby(['agency_subelement_code', 'agency_subelement'])
    .mean()
    .round(0)
    .reset_index()
    .rename(columns={'count': 'avg_headcount_pre_doge'})
)
pre_doge_size.sort_values('avg_headcount_pre_doge', ascending=False).head(20)

In [ ]:
# % young workers (20-29) per agency in pre-DOGE period
pre_doge_total = pre_doge.groupby('agency_subelement_code')['count'].sum().rename('total')

young = (
    pre_doge[pre_doge['age_bracket'].isin(['20-24', '25-29'])]
    .groupby('agency_subelement_code')['count'].sum()
    .rename('young_count')
)

pct_young = (young / pre_doge_total * 100).round(1).reset_index()
pct_young.columns = ['agency_subelement_code', 'pct_young_pre_doge']
pct_young.sort_values('pct_young_pre_doge', ascending=False).head(20)

In [ ]:
# % non-tenured (tenure_code 0 = no tenure, 2 = probationary) per agency pre-DOGE
# Tenure codes: 1=permanent career, 2=career conditional (probationary), 3=term/temp, 0=excepted indefinite
non_tenured = (
    pre_doge[pre_doge['tenure_code'].isin(['2', '3', 2.0, 3.0])]
    .groupby('agency_subelement_code')['count'].sum()
    .rename('non_tenured_count')
)

pct_non_tenured = (non_tenured / pre_doge_total * 100).round(1).reset_index()
pct_non_tenured.columns = ['agency_subelement_code', 'pct_non_tenured_pre_doge']
pct_non_tenured.sort_values('pct_non_tenured_pre_doge', ascending=False).head(20)

In [ ]:
# % temporary/term appointments (appointment_type_code 10=career, 15=career cond, 20=temp, 30=excepted, etc.)
# Temporary = codes that aren't career or career-conditional
temp_codes = ['20', '30', '38', '50', '55']  # temp, excepted, vet recruitment, etc.

temp_appt = (
    pre_doge[pre_doge['appointment_type_code'].isin(temp_codes)]
    .groupby('agency_subelement_code')['count'].sum()
    .rename('temp_count')
)

pct_temp = (temp_appt / pre_doge_total * 100).round(1).reset_index()
pct_temp.columns = ['agency_subelement_code', 'pct_temp_pre_doge']
pct_temp.sort_values('pct_temp_pre_doge', ascending=False).head(20)

In [ ]:
# Build combined pre-DOGE agency profile
agency_profile = (
    pre_doge_size
    .merge(pct_young, on='agency_subelement_code', how='left')
    .merge(pct_non_tenured, on='agency_subelement_code', how='left')
    .merge(pct_temp, on='agency_subelement_code', how='left')
)

# Limit to agencies with meaningful size (>=100 avg headcount)
agency_profile = agency_profile[agency_profile['avg_headcount_pre_doge'] >= 100]
agency_profile.sort_values('avg_headcount_pre_doge', ascending=False).head(20)

## Net Headcount Change: Pre-DOGE vs Now
Compare Jan–Jun 2025 average headcount to Jan–Feb 2026 average to identify agencies with workforce collapse.

In [ ]:
# Post-DOGE headcount: Jan-Feb 2026
post_doge = e_df[(e_df['year'] == 2026)].copy()

post_doge_size = (
    post_doge.groupby(['agency_subelement_code', 'agency_subelement', 'ym'])['count']
    .sum()
    .groupby(['agency_subelement_code', 'agency_subelement'])
    .mean()
    .round(0)
    .reset_index()
    .rename(columns={'count': 'avg_headcount_2026'})
)

# Join and compute change
headcount_change = pre_doge_size.merge(
    post_doge_size[['agency_subelement_code', 'avg_headcount_2026']],
    on='agency_subelement_code',
    how='inner'
)

headcount_change['abs_change'] = headcount_change['avg_headcount_2026'] - headcount_change['avg_headcount_pre_doge']
headcount_change['pct_change'] = (headcount_change['abs_change'] / headcount_change['avg_headcount_pre_doge'] * 100).round(1)

# Largest absolute losses
headcount_change.sort_values('abs_change').head(25)

In [ ]:
# Largest % losses (agencies with >=500 pre-DOGE headcount to filter noise)
headcount_change[
    headcount_change['avg_headcount_pre_doge'] >= 500
].sort_values('pct_change').head(25)

## Separation Rates: Joining Separations to Employment
Compute monthly involuntary separation rate per agency = involuntary separations / beginning-of-month headcount.

In [ ]:
# Load separations data
s_df = load_r2_data("separations")

INVOLUNTARY = {"SH", "SJ"}
s_df["sep_class"] = s_df["separation_category_code"].apply(
    lambda c: "involuntary" if c in INVOLUNTARY else ("voluntary" if c in {"SC","SD","SE","SG","SA","SB"} else "other")
)
s_df['ym'] = s_df['year'].astype(str) + '-' + s_df['month'].astype(str).str.zfill(2)

print(f"Separations loaded: {s_df.shape[0]:,} rows")

In [ ]:
# Involuntary separations per agency per month
inv_by_agency_month = (
    s_df[s_df['sep_class'] == 'involuntary']
    .groupby(['ym', 'agency_subelement_code'])['count']
    .sum()
    .reset_index()
    .rename(columns={'count': 'inv_sep_count'})
)

# Employment headcount per agency per month
emp_by_agency_month = (
    e_df.groupby(['ym', 'agency_subelement_code'])['count']
    .sum()
    .reset_index()
    .rename(columns={'count': 'headcount'})
)

# Join
rates = emp_by_agency_month.merge(inv_by_agency_month, on=['ym', 'agency_subelement_code'], how='left')
rates['inv_sep_count'] = rates['inv_sep_count'].fillna(0)
rates['inv_sep_rate_pct'] = (rates['inv_sep_count'] / rates['headcount'] * 100).round(3)

print(f"Rate table shape: {rates.shape}")
rates.head()

In [ ]:
# Top agencies by peak involuntary separation rate during DOGE period (Jul 2025 - Feb 2026)
doge_rates = rates[rates['ym'] >= '2025-07'].copy()

peak_rates = (
    doge_rates.groupby('agency_subelement_code')
    .agg(
        max_inv_rate=('inv_sep_rate_pct', 'max'),
        total_inv_seps=('inv_sep_count', 'sum'),
        avg_headcount=('headcount', 'mean')
    )
    .reset_index()
)

# Add agency name
agency_names = e_df[['agency_subelement_code', 'agency_subelement']].drop_duplicates()
peak_rates = peak_rates.merge(agency_names, on='agency_subelement_code', how='left')

peak_rates[
    peak_rates['avg_headcount'] >= 500
].sort_values('max_inv_rate', ascending=False).head(20)

In [ ]:
# Monthly involuntary rate for top DOGE-affected agencies
top_doge_agencies = [
    'AM00',  # USAID
    'HE36',  # FDA
    'HE38',  # NIH
    'HE39',  # CDC
    'ST00',  # State Dept
    'DD48',  # Defense Human Resources Activity
    'SB00',  # Small Business Administration
    'HE34',  # HRSA
]

top_rates = (
    rates[rates['agency_subelement_code'].isin(top_doge_agencies)]
    .merge(agency_names, on='agency_subelement_code', how='left')
    .pivot_table(index='ym', columns='agency_subelement', values='inv_sep_rate_pct', fill_value=0)
    .sort_index()
)
top_rates

## Workforce Profile Shift Over Time
Are agencies getting older, more tenured, or less tenured over time? Track composition changes.

In [ ]:
# % young workers (20-29) across entire federal workforce per month
monthly_young = (
    e_df.groupby(['ym', 'age_bracket'])['count']
    .sum()
    .reset_index()
)
monthly_total_map = monthly_total.set_index('ym')['count'].to_dict()
monthly_young['total'] = monthly_young['ym'].map(monthly_total_map)
monthly_young['pct'] = (monthly_young['count'] / monthly_young['total'] * 100).round(2)

young_trend = (
    monthly_young[monthly_young['age_bracket'].isin(['20-24', '25-29'])]
    .groupby('ym')['pct']
    .sum()
    .reset_index()
    .rename(columns={'pct': 'pct_young_20_29'})
    .sort_values('ym')
)
young_trend

In [ ]:
# % career (tenure_code 1) vs non-career across entire workforce per month
monthly_tenure = (
    e_df.groupby(['ym', 'tenure_code'])['count']
    .sum()
    .reset_index()
)
monthly_tenure['total'] = monthly_tenure['ym'].map(monthly_total_map)
monthly_tenure['pct'] = (monthly_tenure['count'] / monthly_tenure['total'] * 100).round(2)

tenure_trend = (
    monthly_tenure
    .pivot_table(index='ym', columns='tenure_code', values='pct', fill_value=0)
    .sort_index()
)
tenure_trend